## Linkedin Ads


#### Useful links:

- [LinkedIn Ad Library API Program](https://www.linkedin.com/ad-library/api)
- [Linkedin Developer Portal](https://www.linkedin.com/developers/)
- [Linkedin Ads API Schema](https://www.linkedin.com/ad-library/api/ads)
- [Linkedin Ad Library Interface](https://www.linkedin.com/ad-library/)


In [ ]:
# Requirements
# %pip install pandas requests python-dotenv rich pydantic

In [1]:
import json
import os
import time
from datetime import datetime, timezone, tzinfo
from typing import Any

import pandas as pd
import requests
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
from pydantic import BaseModel, JsonValue, TypeAdapter
from rich import print as pprint

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

DATA_DIR = "../../data"
BASE_NAME = "eu-linkedin-ads"


# File management
def save_json(question_info: str, json_data: list[Any] | dict[str, Any]):
    now_str = datetime.now(tz=timezone.utc).isoformat()
    filepath = f"{DATA_DIR}/{BASE_NAME}-{question_info}-{now_str}.json"
    print(f"----> {filepath}")
    with open(filepath, "w") as fout:
        json.dump(json_data, fout, indent=4, ensure_ascii=False)


# Environment variables
def get_headers():
    load_dotenv()
    return {
        "X-RestLi-Protocol-Version": "2.0.0",
        "LinkedIn-Version": "202503",
        "Authorization": f"Bearer {os.getenv('LINKEDIN_ACCESS_TOKEN')}",
    }

#### API Connection


In [2]:
ENDPOINT: str = "https://api.linkedin.com/rest/adLibrary"


class Countries(BaseModel):
    codes: list[str]

    def __str__(self) -> str:
        urns = ",".join([f"urn:li:country:{code}" for code in self.codes])
        return f"(value:List({urns.replace(':', '%3A')}))"


def make_datetime(
    obj: int | float | str | datetime | None = None,
    default_dt: datetime | None = None,
    default_tz: tzinfo = timezone.utc,
) -> datetime:
    if obj is None:
        assert default_dt, "Must pass an object to transform or a default value"
        obj = default_dt
    if isinstance(obj, int | float):
        obj = datetime.fromtimestamp(obj)
    if isinstance(obj, str):
        obj = datetime.fromisoformat(obj)
    return (
        obj.astimezone(default_tz)
        if obj.tzinfo
        else obj.replace(tzinfo=default_tz)
    )


class DateRange(BaseModel):
    start: datetime
    end: datetime

    def __init__(
        self,
        start: str | datetime | None = None,
        end: str | datetime | None = None,
    ) -> None:
        now = datetime.now(tz=timezone.utc)
        dt_start = make_datetime(start, default_dt=now - relativedelta(years=1))
        dt_end = make_datetime(end, default_dt=now - relativedelta(days=1))
        super().__init__(start=dt_start, end=dt_end)

    def __str__(self) -> str:
        """Maximum range: `Between yesterday and today minus 1 year.`"""
        range_start = self.start.strftime(r"(start:(day:%d,month:%m,year:%Y)")
        range_end = self.end.strftime(r"end:(day:%d,month:%m,year:%Y))")
        return ",".join((range_start, range_end))


class Params(BaseModel):
    dateRange: DateRange = DateRange()
    countries: Countries | list[str] | None = None
    advertiser: str | None = None
    keyword: str | None = None
    start: int | str = 0
    count: int | str = 25

    def model_post_init(self, context: Any) -> None:
        if isinstance(codes := self.countries, list):
            self.countries = Countries(codes=codes)
        return super().model_post_init(context)

    @property
    def url(self) -> str:
        items = {"q": "criteria", **self.__dict__}.items()
        items = [f"{k}={v}" for k, v in items if v is not None]
        return ENDPOINT.strip("/") + "?" + "&".join(items).replace(" ", "%20")


def forcetype[T](x: Any, type_: type[T]) -> T:
    return TypeAdapter(type_).validate_python(x, strict=True)


def get_response(
    params: Params | None = None,
    api_url: str | None = None,
    save_as: str | None = None,
    show_df: bool = True,
    sleep: int | float = 1,
) -> dict[str, JsonValue]:
    api_url = api_url or (params or Params()).url
    time.sleep(sleep)
    print(f"[GET] {api_url}")
    resp = requests.get(api_url, headers=get_headers(), timeout=30)
    data = resp.json() if resp and resp.ok else json.loads(resp.text)
    data: dict[str, JsonValue] = forcetype(data, dict[str, JsonValue])
    if save_as:
        save_json(save_as, data)
    if "elements" not in data:
        pprint(data)
        return data
    if show_df:
        display(pd.json_normalize(get_elements(data))[:3].T)
    return data


def get_elements(response: dict[str, JsonValue]) -> list[dict[str, Any]]:
    return forcetype(response.get("elements"), list[dict[str, JsonValue]])

### Special Criteria


**SC1: Does the platform provide an API to access its ad repository and extract data on advertising content?**

_This item verifies whether the platform provides an ad repository API with at least one endpoint for programmatically extracting advertising data. Full availability is confirmed when the API returns information on ads across all categories. The assessment should confirm that the endpoint allows the retrieval and storage of ad data without requiring privileged or internal access beyond standard developer registration._


In [3]:
# Yes
data = get_response(
    Params(countries=["DE", "FR"], keyword="Greenwashing"),
    save_as="question-sc1-data",
)

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&keyword=Greenwashing&start=0&count=25
----> ../../data/eu-linkedin-ads-question-sc1-data-2025-11-19T17:30:34.695021+00:00.json


,0,1,2
isRestricted,False,False,False
adUrl,https://www.linkedin.com/ad-library/detail/910...,https://www.linkedin.com/ad-library/detail/268...,https://www.linkedin.com/ad-library/detail/910...
details.advertiser.adPayer,LIBEO CAPITAL,The Digitale GmbH,LIBEO CAPITAL
details.advertiser.advertiserName,Antoine Leurent,BIOFACH,Antoine Leurent
details.advertiser.advertiserUrl,https://www.linkedin.com/in/a-leurent,https://www.linkedin.com/company/41207193,https://www.linkedin.com/in/a-leurent
details.adStatistics.latestImpressionAt,1763335760792,1763317178792,1763333196766
details.adStatistics.impressionsDistributionByCountry,"[{'country': 'urn:li:country:GB', 'impressionP...","[{'country': 'urn:li:country:AF', 'impressionP...","[{'country': 'urn:li:country:GB', 'impressionP..."
details.adStatistics.totalImpressions.from,5000,10000,5000
details.adStatistics.totalImpressions.to,10000,50000,10000
details.adStatistics.firstImpressionAt,1763020461187,1762513963403,1763021177151


**SC3: Can data from both active and inactive ads be extracted?**

_This item verifies whether the platform allows the extraction of ad data through either the GUI or the API, from at least one day after publication to at least one year prior. Full availability is confirmed when both active and inactive ad data are delivered across all ad categories. The assessment should test the interface and endpoints to confirm whether both active and inactive ads can be retrieved._


In [4]:
# Yes. Data with latestImpressionAt before six months ago:
now = datetime.now(tz=timezone.utc)
six_months_ago = now - relativedelta(months=6)
response = get_response(
    Params(countries=["DE", "FR"], dateRange=DateRange(end=six_months_ago)),
    save_as="question-sc3-data",
    show_df=False,
)
elements: list[dict[str, Any]] = []
for elem in get_elements(response):
    latest_imp = elem["details"]["adStatistics"]["latestImpressionAt"]
    latest_imp_dt = make_datetime(latest_imp / 1000)
    if latest_imp_dt and latest_imp_dt < six_months_ago:
        print("Latest impression:", latest_imp_dt.isoformat())
        elements.append(elem)
pd.json_normalize(elements).T

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:19,month:05,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&start=0&count=25
----> ../../data/eu-linkedin-ads-question-sc3-data-2025-11-19T17:30:50.535813+00:00.json


""


### ACCESSIBILITY


**OC3: Can the requested data be extracted directly from the ad repository response?**

This item verifies whether the platform ad repository returns structured data on ad creatives and authorship directly in the response, rather than providing a link that redirects to the data. Audiovisual media files and data (e.g., images, videos, and audio) should not be considered when assessing this item. The assessment should examine sample data responses from both the ad repository GUI and API to confirm that the requested public data is included in the returned payload.\*


* **No** (for API). The endpoint returns only the ad's URL, the ad's statistics and data about the advertiser, including `advertiserName`, `advertiserUrl` and `adPayer`, but neither the ad's text nor publication date.

**OC5: Can data from an individual ad be retrieved from the platform?**

_This item verifies whether it is possible to retrieve data from a specific advertisement on the ad repository using a unique identifier, rather than relying on search terms or other parameters and filters. The assessment should review the ad repository documentation and test available features to confirm that an individual ad can be retrieved directly by its unique identifier._


* **No** (for API). There is no endpoint for that. To get data from an individual ad it must be requested outside the API via browser or HTTP request.

**OC6: Can data from ads served by a specific advertiser be retrieved from the platform?**

_This item verifies whether it is possible to retrieve data from ads run by a specific advertiser, via their username or unique identifier. The assessment should review the ad repository documentation and test any available feature to retrieve data from an individual advertiser._


In [5]:
# No (for API). The search for advertisers only retrieves data that contains the
# searched keyword instead of filtering data from a specific advertiser via
# their username or unique identifier.
response = get_response(
    Params(countries=["DE", "FR"], advertiser="Shell"),
    save_as="question-oc06-data",
)

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&advertiser=Shell&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc06-data-2025-11-19T17:30:58.148402+00:00.json


,0,1,2
isRestricted,False,False,False
adUrl,https://www.linkedin.com/ad-library/detail/906...,https://www.linkedin.com/ad-library/detail/906...,https://www.linkedin.com/ad-library/detail/906...
details.advertiser.adPayer,RCKT GMBH,RCKT GMBH,RCKT GMBH
details.advertiser.advertiserName,Shell Fleet Solutions,Shell Fleet Solutions,Shell Fleet Solutions
details.advertiser.advertiserUrl,https://www.linkedin.com/company/76516632,https://www.linkedin.com/company/76516632,https://www.linkedin.com/company/76516632
details.adStatistics.latestImpressionAt,1763331758689,1763331026490,1763328432718
details.adStatistics.impressionsDistributionByCountry,"[{'country': 'urn:li:country:DE', 'impressionP...","[{'country': 'urn:li:country:AT', 'impressionP...","[{'country': 'urn:li:country:DE', 'impressionP..."
details.adStatistics.totalImpressions.from,0,0,1000
details.adStatistics.totalImpressions.to,1000,1000,5000
details.adStatistics.firstImpressionAt,1762954412445,1762955012680,1762954987649


**OC7: Can ad data be retrieved from the platform using search terms?**

_This item verifies whether data on ad campaigns can be retrieved via individual or combined search terms, enabling the creation of a dataset composed of ads that mention those terms. The assessment should test search-related features to confirm that queries using keywords return relevant ad campaign data._


In [6]:
# Yes
response = get_response(
    Params(countries=["DE", "FR"], keyword="Greenwashing"),
    save_as="question-oc07-data",
)

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&keyword=Greenwashing&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc07-data-2025-11-19T17:31:04.122418+00:00.json


,0,1,2
isRestricted,False,False,False
adUrl,https://www.linkedin.com/ad-library/detail/910...,https://www.linkedin.com/ad-library/detail/268...,https://www.linkedin.com/ad-library/detail/910...
details.advertiser.adPayer,LIBEO CAPITAL,The Digitale GmbH,LIBEO CAPITAL
details.advertiser.advertiserName,Antoine Leurent,BIOFACH,Antoine Leurent
details.advertiser.advertiserUrl,https://www.linkedin.com/in/a-leurent,https://www.linkedin.com/company/41207193,https://www.linkedin.com/in/a-leurent
details.adStatistics.latestImpressionAt,1763335760792,1763317178792,1763333196766
details.adStatistics.impressionsDistributionByCountry,"[{'country': 'urn:li:country:GB', 'impressionP...","[{'country': 'urn:li:country:AF', 'impressionP...","[{'country': 'urn:li:country:GB', 'impressionP..."
details.adStatistics.totalImpressions.from,5000,10000,5000
details.adStatistics.totalImpressions.to,10000,50000,10000
details.adStatistics.firstImpressionAt,1763020461187,1762513963403,1763021177151


**OC8: Does the platform use locale-neutral data representations?**

_This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are provided in a locale-neutral format, or, if that is not possible, whether relevant locale metadata is included. The assessment should review the ad repository documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata._


In [7]:
# No, because the timestamps are all numeric
# (not in isoformat, and timezone-unaware).
response = get_response(
    Params(countries=["DE", "FR"]), save_as="question-oc08-data", show_df=False
)
elements = get_elements(response)
for elem in elements[:5]:
    statistics = elem["details"]["adStatistics"]
    print(
        "firstImpressionAt:",
        statistics["firstImpressionAt"],
        "| latestImpressionAt:",
        statistics["latestImpressionAt"],
    )

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc08-data-2025-11-19T17:31:10.782299+00:00.json
firstImpressionAt: 1763336706521 | latestImpressionAt: 1763337597116
firstImpressionAt: 1763332121656 | latestImpressionAt: 1763333064881
firstImpressionAt: 1763330500440 | latestImpressionAt: 1763337492289
firstImpressionAt: 1763331627770 | latestImpressionAt: 1763336951670
firstImpressionAt: 1763336305939 | latestImpressionAt: 1763337596392


### COMPLETENESS


**OC9: Does the platform provide data that allows the identification of advertisers who ran ads?**

_This item verifies whether the platform discloses information on the advertisers responsible for the identified ads. The assessment should confirm whether the advertiser’s page name, URL, and unique identifier can be retrieved._


In [8]:
# Yes
response = get_response(
    Params(countries=["DE", "FR"], keyword="Greenwashing"),
    save_as="question-oc09-data",
    show_df=False,
)
# Showing details about advertisers.
elements = get_elements(response)
display(pd.json_normalize(elements).filter(like="advertiser"))

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&keyword=Greenwashing&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc09-data-2025-11-19T17:31:23.572851+00:00.json


,details.advertiser.adPayer,details.advertiser.advertiserName,details.advertiser.advertiserUrl
0,LIBEO CAPITAL,Antoine Leurent,https://www.linkedin.com/in/a-leurent
1,The Digitale GmbH,BIOFACH,https://www.linkedin.com/company/41207193
2,LIBEO CAPITAL,Antoine Leurent,https://www.linkedin.com/in/a-leurent
3,The Digitale GmbH,BIOFACH,https://www.linkedin.com/company/41207193
4,STUDIO BIRTHPLACE PTE LTD,Greenpeace,https://www.linkedin.com/company/7458
5,DNV Business Assurance Norway AS,DNV - Supply Chain & Product Assurance,https://www.linkedin.com/company/77666243
6,DLA PIPER UK LLP,DLA Piper,https://www.linkedin.com/company/4422
7,Clyde & Co LLP,Clyde & Co,https://www.linkedin.com/company/284053
8,Food Nature Climate Dialogue,Food Nature Climate Dialogue,https://www.linkedin.com/company/64740854
9,PANORAMA300 GmbH & Co.KG,Projekt Zukunft Berlin,https://www.linkedin.com/company/18457256


**OC10: Does the platform provide data on the funders who paid for ads?**

_This item verifies whether the platform provides data on who paid for ad campaigns. The assessment should confirm whether any sponsor information can be retrieved._


In [9]:
# Yes
response = get_response(
    Params(countries=["DE", "FR"], keyword="Greenwashing"),
    save_as="question-oc10-data",
    show_df=False,
)
# Showing only the adPayer field.
elements = get_elements(response)
display(pd.json_normalize(elements).filter(like="adPayer"))

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&keyword=Greenwashing&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc10-data-2025-11-19T17:31:31.089031+00:00.json


,details.advertiser.adPayer
0,LIBEO CAPITAL
1,The Digitale GmbH
2,LIBEO CAPITAL
3,The Digitale GmbH
4,STUDIO BIRTHPLACE PTE LTD
5,DNV Business Assurance Norway AS
6,DLA PIPER UK LLP
7,Clyde & Co LLP
8,Food Nature Climate Dialogue
9,PANORAMA300 GmbH & Co.KG


**OC11: Does the platform provide data on the period during which ads were served?**

_This item verifies whether the platform provides data on the days on which ad campaigns ran. The assessment should review ad records to confirm that campaigns include start and end dates (or equivalent temporal markers) covering their active period_


In [10]:
# Yes, if we count on documentation stating that all timestamps are UTC-based.
response = get_response(
    Params(
        countries=["DE", "FR"],
        dateRange=DateRange(
            end=datetime.now(timezone.utc) - relativedelta(months=3)
        ),
    ),
    save_as="question-oc11-data",
    show_df=False,
)
elements = get_elements(response)
for elem in elements[:5]:
    statistics = elem["details"]["adStatistics"]
    first = make_datetime(statistics["firstImpressionAt"] / 1000)
    latest = make_datetime(statistics["latestImpressionAt"] / 1000)
    print(
        "firstImpressionAt:",
        first.isoformat(),
        "| latestImpressionAt:",
        latest.isoformat(),
    )

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:19,month:08,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc11-data-2025-11-19T17:31:43.374094+00:00.json
firstImpressionAt: 2025-08-18T20:45:59.758000+00:00 | latestImpressionAt: 2025-10-29T21:54:12.283000+00:00
firstImpressionAt: 2025-08-18T20:12:54.731000+00:00 | latestImpressionAt: 2025-08-18T20:26:52.959000+00:00
firstImpressionAt: 2025-08-18T20:47:02.485000+00:00 | latestImpressionAt: 2025-10-26T21:54:12.064000+00:00
firstImpressionAt: 2025-08-18T20:47:03.951000+00:00 | latestImpressionAt: 2025-10-31T09:14:37.162000+00:00
firstImpressionAt: 2025-08-18T20:46:25.422000+00:00 | latestImpressionAt: 2025-10-29T06:45:46.968000+00:00


**OC12: Does the platform provide data on user engagement with ads?**

_This item verifies whether the platform provides data on the total number of user interactions with ad campaigns (e.g., likes, comments, shares, clicks). The assessment should review ad records to confirm that engagement metrics are available and clearly linked to each campaign._


In [11]:
# Yes, partially.
response = get_response(
    Params(countries=["DE", "FR"]), save_as="question-oc12-data", show_df=False
)
elements = get_elements(response)
pd.json_normalize(elements).filter(like="adStatistics")[:3].T

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc12-data-2025-11-19T17:31:57.121374+00:00.json


,0,1,2
details.adStatistics.latestImpressionAt,1763337597116,1763333064881,1763337492289
details.adStatistics.impressionsDistributionByCountry,[],[],[]
details.adStatistics.totalImpressions.from,0,0,0
details.adStatistics.totalImpressions.to,1000,1000,1000
details.adStatistics.firstImpressionAt,1763336706521,1763332121656,1763330500440


**OC13: Does the platform indicate whether ads were placed by verified or unverified advertisers?**

_This item verifies whether the platform clearly indicates whether advertisers were verified at the time their ads were served. The assessment should review ad records to confirm that a verification status field is present._


* **No**. The only fields about advertisers in response schema are: `advertiserName`, `advertiserUrl` and `adPayer`.
* Ref.: https://www.linkedin.com/ad-library/api/ads

### COMPLIANCE


**OC14: Does the platform flag ads that were removed due to violations of its guidelines or relevant legislation?**

_This item verifies whether the platform indicates when an ad has been moderated. At a minimum, the platform should provide the reason for removal and the date. The assessment should review ad records to confirm that moderated ads are flagged and that the corresponding moderation details are clearly documented._


* **Yes**. The response has the field `isRestricted: True` for flagged ads, and the details are stored at the field `restrictionDetails`.

In [ ]:
# THIS DEMONSTRATION IS NOT REPRODUCIBLE (restricted ads are rare).
# DO NOT RUN THIS CELL AGAIN.
now = datetime.now(tz=timezone.utc)
six_months_ago = now - relativedelta(months=6)
response = get_response(
    Params(countries=["DE", "FR"], dateRange=DateRange(end=six_months_ago)),
    save_as="question-oc14-data",
    show_df=False,
)
elements = get_elements(response)
display(pd.json_normalize([ad for ad in elements if ad["isRestricted"]]).T)
# [GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:18,month:11,year:2024),end:(day:18,month:05,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&start=0&count=25
# ----> ../../data/eu-linkedin-ads-question-oc14-data-2025-11-18T17:55:31.844420+00:00.json
# adUrl: https://www.linkedin.com/ad-library/detail/673001394

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:18,month:11,year:2024),end:(day:18,month:05,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc14-data-2025-11-18T17:55:31.844420+00:00.json


,0
adUrl,https://www.linkedin.com/ad-library/detail/673...
isRestricted,True
restrictionDetails,This ad was removed because it violated Linked...
details.advertiser.advertiserName,InstantDigi
details.advertiser.advertiserUrl,https://www.linkedin.com/company/103562336
details.adStatistics.latestImpressionAt,1747526365994
details.adStatistics.impressionsDistributionByCountry,"[{'country': 'urn:li:country:AD', 'impressionP..."
details.adStatistics.totalImpressions.from,10000
details.adStatistics.totalImpressions.to,50000
details.adStatistics.firstImpressionAt,1747519622366


Saving restricted ad basic data:

```json
{
    "adUrl": "https://www.linkedin.com/ad-library/detail/673001394",
    "details": {
        "advertiser": {
            "advertiserName": "InstantDigi",
            "advertiserUrl": "https://www.linkedin.com/company/103562336"
        },
        "type": "SPONSORED_STATUS_UPDATE"
    },
    "isRestricted": true,
    "restrictionDetails": "This ad was removed because it violated LinkedIn's Advertising Policy. Our automated systems removed this ad after it was reported to LinkedIn."
}
```

**OC15: Does the platform indicate whether ad creatives were generated using artificial intelligence?**

_This item verifies whether the platform flags ad campaigns in which AI played a role in producing the content. The assessment should review ad records to confirm the presence of a field or label indicating the use of AI in ad production._


* **No**. There are no mentions to AI anywhere.
* Ref.: https://www.linkedin.com/ad-library/api/ads

### CONSISTENCY


**OC23: Does the data retrieved by the API reflect what is displayed on the platform’s ad repository GUI?**

_This item verifies whether the data returned by the platform’s ad repository API corresponds to the information displayed on its GUI in all its levels of detail. It should be possible to identify in the API response information such as authorship, complete content, and other campaigning information (e.g., amount spent, impressions reached). The assessment should compare API responses with the GUI to confirm that key elements—such as authorship, full content, and campaign information (e.g., spending, impressions)—are consistently included._


* **No**. The GUI search page shows a preview of each ad's media and text, while the API returns no information whatsoever about each ad's content, only its URL and advertiser data.

**OC24: Are the results returned by the platform consistently reproducible?**

_This item verifies whether data accessed and extracted via the platform’s ad repository at a given time is consistent with other collections performed similarly, including cases where content was deleted in the interim. The assessment should perform repeated queries to confirm reproducibility of results._


Test instructions: Please, develop a test that collects ads for 5 times using the same request parameters and the same endpoint. Save each response in separate files.


In [12]:
# Yes
total_runs = 5
responses = []
for index in range(1, total_runs + 1):
    resp = get_response(
        Params(countries=["DE", "FR"], advertiser="Shell"),
        save_as=f"question-oc24-run-{index}",
        show_df=False,
        sleep=7,
    )
    responses.append(resp)
comparison = (
    responses[0] == responses[1] == responses[2] == responses[3] == responses[4]
)
pprint(f"All equal: {comparison}")

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&advertiser=Shell&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc24-run-1-2025-11-19T17:32:22.333427+00:00.json
[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&advertiser=Shell&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc24-run-2-2025-11-19T17:32:30.598654+00:00.json
[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&advertiser=Shell&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc24-run-3-2025-11-19T17:32:38.811762+00:00.json
[GET

All equal: True

**OC25: Is the data returned by the platform consistent with the parameters and filters used in the request?**

_This item verifies whether the data retrieved through the ad repository accurately reflects the parameters and filters specified at the time of the request. The assessment should run test queries with different filters to confirm that results consistently match the requested conditions._


Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files.


* **No**. The information cannot be verified through the API response alone. Only the advertiser details and adStatistics are returned at the API response, so these are the only fields that could be verified programatically without scraping the ad page itself with the `adUrl` provided.
* Refs.:
    - https://www.linkedin.com/ad-library/api/ads
    - https://www.linkedin.com/help/lms/answer/a1620070

### RELEVANCE


**OC26: Does the platform allow the use of temporal filters to retrieve data on ads?**

_This item verifies whether the ad repository allows filtering ad campaign data based on the time period during which the ads were served. The assessment should test queries with temporal filters to confirm that results accurately reflect the specified date ranges._


In [13]:
# Yes
now = datetime.now(tz=timezone.utc)
ten_months_ago = now - relativedelta(months=10)
six_months_ago = now - relativedelta(months=6)
response = get_response(
    params=Params(
        countries=["DE", "FR"],
        dateRange=DateRange(start=ten_months_ago, end=six_months_ago),
    ),
    save_as="question-oc26-data",
    show_df=False,
)
elements = get_elements(response)
for elem in elements:
    statistics = elem["details"]["adStatistics"]
    first = make_datetime(statistics["firstImpressionAt"] / 1000)
    between = ten_months_ago < first < six_months_ago
    print(
        "firstImpressionAt:",
        first.isoformat(),
        f"| Between ten and six months ago: {between}",
    )

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:01,year:2025),end:(day:19,month:05,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc26-data-2025-11-19T17:33:20.723621+00:00.json
firstImpressionAt: 2025-05-18T19:14:59.982000+00:00 | Between ten and six months ago: True
firstImpressionAt: 2025-05-18T19:03:31.374000+00:00 | Between ten and six months ago: True
firstImpressionAt: 2025-05-18T18:46:29.972000+00:00 | Between ten and six months ago: True
firstImpressionAt: 2025-05-18T19:45:39.296000+00:00 | Between ten and six months ago: True
firstImpressionAt: 2025-05-18T18:46:07.259000+00:00 | Between ten and six months ago: True
firstImpressionAt: 2025-05-18T18:46:15.145000+00:00 | Between ten and six months ago: True
firstImpressionAt: 2025-05-18T19:11:27.727000+00:00 | Between ten and six months ago: True
firstImpressionAt: 2025-05-18T18:59:55.095000+00:

**OC27: Does the platform allow filtering advertising data by ad category?**

_This item verifies whether the ad repository allows filtering ad campaign data based on any possible categories assigned at the time of ad creation. The assessment should run test queries with category filters to confirm that results align with the selected classifications._


* **No**. There is no category parameter at the request schema.
* Ref.: https://www.linkedin.com/ad-library/api/ads

**OC28: Does the platform allow filtering advertising data by geographic location?**

*This item assesses whether the ad repository allows filtering data by one or more subnational geographic locations where the ads were served. The assessment should test queries with location filters to confirm that results match the specified areas.*



* **No**. There is no information about subnational locations at the API.
* Ref.: https://www.linkedin.com/ad-library/api/ads

### ACCURACY


**OC29: Does the platform provide age and gender data on the audiences of ads?**

_This item verifies whether the platform provides data on the age and gender of audiences reached. The assessment should review the ad records to confirm that these breakdowns are available and consistently reported_


* **No**. There is neither age nor gender parameters at the request schema.
* Ref.: https://www.linkedin.com/ad-library/api/ads

**OC30: Does the platform provide subnational geographic data on the audience reached by ads?**

_This item verifies whether the platform provides data on the subnational geographic location of audiences reached. The assessment should review the ad records to confirm that such location breakdowns are available and consistently reported._


* **No**. There is no subnational option at the response schema.
* Ref.: https://www.linkedin.com/ad-library/api/ads

**OC31: Does the platform include data on audience targeting criteria defined by advertisers?**

_This item verifies whether the platform provides data on audience targeting criteria defined by the advertiser when publishing ads (e.g., demographic and geographic segments, as well as interests, attitudes, behaviors, and keywords). The assessment should review ad records to confirm that these targeting parameters are available and consistently reported._


In [14]:
# Yes
response = get_response(
    Params(countries=["DE", "FR"]), save_as="question-oc31-data", show_df=False
)
elements = get_elements(response)
for index in range(3):
    pprint(f"adUrl: {elements[index]['adUrl']}")
    display(pd.json_normalize(elements[index]["details"]["adTargeting"]))

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc31-data-2025-11-19T17:33:31.972925+00:00.json


adUrl: https://www.linkedin.com/ad-library/detail/916311084

,includedSegments,facetName,excludedSegments,isExcluded,isIncluded
0,[Français],Language,[],False,True
1,"[Clisson, Roussay, Le Bignon, Nantes et périph...",Location,[],False,True


adUrl: https://www.linkedin.com/ad-library/detail/916228394

,includedSegments,facetName,excludedSegments,isExcluded,isIncluded
0,[English],Language,[],False,True
1,"[Île-de-France, France, Greater Lyon Area, Pro...",Location,[],False,True
2,[],Job,[],False,True


adUrl: https://www.linkedin.com/ad-library/detail/933860473

,includedSegments,facetName,excludedSegments,isExcluded,isIncluded
0,[English],Language,[],False,True
1,[Germany],Location,[],False,True
2,[],Company,[],True,True


**OC32: Does the platform provide granular volume ranges for ad impressions?**

_This item verifies whether the ad repository provides impression values for ads, using ranges that closely approximate the actual numbers. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to 1,000 impressions should be displayed in intervals no larger than 100; between 1,000 and 10,000 in intervals no larger than 1,000; between 10,000 and 100,000 in intervals no larger than 10,000; between 100,000 and 1 million or above, in intervals no larger than 100,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces._


In [15]:
# No. Only "totalImpressions" are provided.
response = get_response(
    Params(countries=["DE", "FR"]), save_as="question-oc32-data", show_df=False
)
elements = get_elements(response)
for index in range(3):
    pprint(f"adUrl: {elements[index]['adUrl']}")
    display(pd.json_normalize(elements[index]["details"]["adStatistics"]))

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:19,month:11,year:2024),end:(day:18,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE,urn%3Ali%3Acountry%3AFR))&start=0&count=25
----> ../../data/eu-linkedin-ads-question-oc32-data-2025-11-19T17:33:51.516448+00:00.json


adUrl: https://www.linkedin.com/ad-library/detail/916311084

,latestImpressionAt,impressionsDistributionByCountry,firstImpressionAt,totalImpressions.from,totalImpressions.to
0,1763337597116,[],1763336706521,0,1000


adUrl: https://www.linkedin.com/ad-library/detail/916228394

,latestImpressionAt,impressionsDistributionByCountry,firstImpressionAt,totalImpressions.from,totalImpressions.to
0,1763333064881,[],1763332121656,0,1000


adUrl: https://www.linkedin.com/ad-library/detail/933860473

,latestImpressionAt,impressionsDistributionByCountry,firstImpressionAt,totalImpressions.from,totalImpressions.to
0,1763337492289,[],1763330500440,0,1000


**OC33: Does the platform provide granular investment ranges for ad spending?**

_This item verifies whether the ad repository provides spending values for ads, using ranges that closely approximate the actual amounts. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to $100 should be displayed in intervals no larger than $10; between $100 and $1,000 in intervals no larger than $100; and between $1,000 and $10,000 in intervals no larger than $1,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces._


* **No**. There is no ad spending informations at the response schema.
* Ref.: https://www.linkedin.com/ad-library/api/ads